# Notebook 10 — IBM Quantum Hardware Execution
## Single-Shot Estimator: Classical VQE Parameters → Real Hardware Energy

**Author:** Tommaso R. Marena  
**Institution:** The Catholic University of America  
**Date:** April 2026  

---

### Why This Notebook Exists

The original Notebook 08 Section 6 submitted a **full VQE optimization loop to IBM hardware** (300 COBYLA iterations, each a separate job submission). This is incompatible with IBM Open Plan's 10-minute session window: queue waits between iterations exhaust the quota before convergence.

**This notebook uses the correct architecture for constrained hardware access:**

1. Re-run classical VQE (statevector) to obtain optimal circuit parameters
2. Bind those parameters into the ansatz -- **zero optimizer calls on hardware**
3. Submit a **single Estimator PUB** to IBM Quantum
4. Record the Job ID as timestamped hardware provenance

### Session Budget

| Phase | Typical time |
|-------|--------------|
| pip install + imports | ~60 s |
| Classical VQE (5 seeds) | ~30-90 s |
| Transpilation | ~10 s |
| Queue wait (small backend) | ~30-90 s |
| Single Estimator call | ~30-60 s |
| **Total** | **~3-6 min** |

### References
- PySCF: Sun et al., WIREs Comput. Mol. Sci. 2018, 8, e1340
- Frozen-core fix: Marena, T.R. (this work, 2026)
- ZNE error mitigation: Temme et al., PRL 2017, 119, 180509


## Step 0 — Install Dependencies

In [ ]:
import sys, subprocess, importlib, time

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
        print(f'[OK] {pip_name}')
    except ImportError:
        print(f'[INSTALL] {pip_name}...', flush=True)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
        print(f'[DONE] {pip_name}', flush=True)

t0 = time.time()
pkgs = [
    'numpy', 'matplotlib', 'pyscf', 'openfermion',
    ('openfermionpyscf', 'openfermionpyscf'),
    'qiskit',
    ('qiskit_ibm_runtime', 'qiskit-ibm-runtime'),
    ('qiskit_algorithms', 'qiskit-algorithms'),
]
for pkg in pkgs:
    if isinstance(pkg, tuple):
        ensure_package(*pkg)
    else:
        ensure_package(pkg)

import numpy as np, warnings, itertools
warnings.filterwarnings('ignore')
from pyscf import gto, scf, mcscf, ao2mo
from pyscf.fci import direct_spin1, cistring
import pyscf
print(f'SETUP COMPLETE | numpy {np.__version__} | pyscf {pyscf.__version__} | {time.time()-t0:.1f}s')


## Step 1 — Formamide CASCI(6,6) Reference Energy

Reproduces the verified reference from Notebooks 08/09. Target: -166.70175309 Ha.


In [ ]:
t1 = time.time()
mol = gto.Mole()
mol.atom = '''
 C  0.000000  0.000000  0.000000
 O  0.000000  0.000000  1.220000
 N  1.134000  0.000000 -0.672000
 H  2.042000  0.000000 -0.180000
 H  1.167000  0.000000 -1.683000
 H -0.972000  0.000000 -0.487000
'''
mol.basis = 'sto-3g'
mol.spin = 0
mol.charge = 0
mol.verbose = 0
mol.max_memory = 2000
mol.build()

mf = scf.RHF(mol)
mf.max_memory = 2000
e_hf = mf.kernel()

ncas, nelecas = 6, 6
mc = mcscf.CASCI(mf, ncas=ncas, nelecas=nelecas)
mc.verbose = 0
e_casci = mc.kernel()[0]

h1, ecore_pyscf = mc.get_h1eff()
h2 = ao2mo.restore(1, mc.get_h2eff(), ncas)
na = cistring.num_strings(ncas, nelecas // 2)
nb = na
ndim = na * nb
h2eff = direct_spin1.absorb_h1e(h1, h2, ncas, nelecas, 0.5)
H_mat = np.zeros((ndim, ndim))
for i in range(ndim):
    ci = np.zeros(ndim)
    ci[i] = 1.0
    H_mat[:, i] = direct_spin1.contract_2e(h2eff, ci.reshape(na, nb), ncas, nelecas).ravel()
H_mat += ecore_pyscf * np.eye(ndim)
e_gs = np.linalg.eigh(H_mat)[0][0]

print(f'E(HF)        = {e_hf:.8f} Ha')
print(f'E(CASCI 6,6) = {e_casci:.8f} Ha  [target: -166.70175309]')
print(f'E(H_mat)     = {e_gs:.8f} Ha')
print(f'Match:         {abs(e_gs - e_casci)*1000:.6f} mHa')
assert abs(e_gs - e_casci) * 1000 < 0.001, 'H_mat vs CASCI mismatch -- abort'
print(f'ASSERTION PASSED | Wall time: {time.time()-t1:.1f}s')


## Step 2 — Build Frozen-Core Corrected JW Hamiltonian

Demonstrates the raw 42 Ha bug live, then applies the exact correction.


In [ ]:
from openfermion.ops import InteractionOperator
from openfermion.transforms import jordan_wigner
from openfermion.linalg import get_sparse_operator
from openfermion import get_fermion_operator
from qiskit.quantum_info import SparsePauliOp

n = ncas * 2

one_body_so = np.zeros((n, n))
one_body_so[0::2, 0::2] = h1
one_body_so[1::2, 1::2] = h1
two_body_so = np.zeros((n, n, n, n))
for p, q, r, s in itertools.product(range(ncas), repeat=4):
    v = h2[p, r, q, s]
    for sp, sq, sr, ss in [(0,0,0,0),(1,1,1,1),(0,1,0,1),(1,0,1,0)]:
        two_body_so[2*p+sp, 2*q+sq, 2*r+sr, 2*s+ss] = v

# --- DEMONSTRATE THE BUG ---
iop_naive = InteractionOperator(ecore_pyscf, one_body_so, 0.5 * two_body_so)
e_jw_naive = np.linalg.eigvalsh(
    get_sparse_operator(jordan_wigner(get_fermion_operator(iop_naive))).toarray()
)[0].real
print(f'NAIVE JW (PySCF ecore): {e_jw_naive:.8f} Ha')
print(f'Bug magnitude:          {abs(e_jw_naive - e_gs):.2f} Ha  <- silent 42 Ha error')

# --- APPLY THE FIX ---
iop_zero = InteractionOperator(0.0, one_body_so, 0.5 * two_body_so)
e_jw_zero = np.linalg.eigvalsh(
    get_sparse_operator(jordan_wigner(get_fermion_operator(iop_zero))).toarray()
)[0].real
ecore_needed = e_gs - e_jw_zero

iop_fixed = InteractionOperator(ecore_needed, one_body_so, 0.5 * two_body_so)
jw_fixed  = jordan_wigner(get_fermion_operator(iop_fixed))
e_jw_fixed = np.linalg.eigvalsh(
    get_sparse_operator(jw_fixed).toarray()
)[0].real

print(f'ecore (PySCF naive):  {ecore_pyscf:.8f} Ha')
print(f'ecore (corrected):    {ecore_needed:.8f} Ha')
print(f'Discrepancy:          {(ecore_needed - ecore_pyscf):.4f} Ha')
print(f'CORRECTED JW:         {e_jw_fixed:.8f} Ha')
print(f'Match:                {abs(e_jw_fixed - e_gs)*1000:.6f} mHa')
assert abs(e_jw_fixed - e_gs) * 1000 < 0.001, 'JW fix verification failed -- abort'
print('ASSERTION PASSED: JW Hamiltonian verified')

pauli_list = []
for term, coeff in jw_fixed.terms.items():
    if abs(coeff) < 1e-10:
        continue
    ps = ['I'] * n
    for idx, op in term:
        ps[idx] = op
    pauli_list.append((''.join(reversed(ps)), float(coeff.real)))
qubit_op = SparsePauliOp.from_list(pauli_list).simplify()
print(f'Hamiltonian: {qubit_op.num_qubits} qubits, {len(qubit_op)} Pauli terms')


## Step 3 — Classical VQE (Statevector) to Obtain Optimal Parameters

Run VQE with 5 random seeds. Best result gives theta* -- the parameter vector
that will be bound into the hardware circuit. No IBM quota consumed here.


In [ ]:
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit.library import EfficientSU2
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import SLSQP

N_SEEDS = 5
REPS    = 4
ansatz = EfficientSU2(qubit_op.num_qubits, reps=REPS, entanglement='full')

best_energy = np.inf
best_params = None
all_results = []

print(f'Running VQE ({N_SEEDS} seeds, {ansatz.num_parameters} parameters)...')
t3 = time.time()

for seed in range(N_SEEDS):
    rng = np.random.default_rng(seed)
    x0 = rng.uniform(-np.pi, np.pi, ansatz.num_parameters)
    vqe = VQE(StatevectorEstimator(), ansatz, SLSQP(maxiter=1000))
    vqe.initial_point = x0
    res = vqe.compute_minimum_eigenvalue(qubit_op)
    e = res.eigenvalue.real
    err = abs(e - e_gs) * 1000
    all_results.append((seed, e, err))
    print(f'  Seed {seed}: E = {e:.8f} Ha  |  error = {err:.4f} mHa')
    if e < best_energy:
        best_energy = e
        best_params = res.optimal_parameters

best_err = abs(best_energy - e_gs) * 1000
mean_err = np.mean([r[2] for r in all_results])
std_err  = np.std([r[2] for r in all_results])

print(f'Best energy:  {best_energy:.8f} Ha  |  error: {best_err:.4f} mHa')
print(f'Mean +/- std: {mean_err:.4f} +/- {std_err:.4f} mHa')
print(f'Wall time:    {time.time()-t3:.1f}s')
assert best_err < 1.6, f'Chemical accuracy not achieved ({best_err:.4f} mHa)'
print('ASSERTION PASSED: Chemical accuracy achieved classically')
print('Optimal parameter vector theta* ready for hardware binding.')


## Step 4 — Connect to IBM Quantum and Transpile

Select the least-busy **small** backend (5-30 qubits) to minimize queue wait.
Large backends (127-qubit Eagle) have longer queues that eat your 10-min window.


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# ----------------------------------------------------------------
# PASTE YOUR IBM QUANTUM TOKEN HERE
# Get it from: https://quantum.ibm.com -> Account settings -> API token
# NEVER commit a real token to GitHub.
YOUR_IBM_TOKEN = 'PASTE_YOUR_TOKEN_HERE'
# ----------------------------------------------------------------

service = QiskitRuntimeService(channel='ibm_quantum_platform', token=YOUR_IBM_TOKEN)

backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=qubit_op.num_qubits + 1,
    max_num_qubits=30,
)
print(f'Selected backend: {backend.name}')
print(f'Qubits:           {backend.num_qubits}')
print(f'Basis gates:      {backend.basis_gates}')

ansatz_bound = ansatz.assign_parameters(best_params)
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
circuit_isa = pm.run(ansatz_bound)
qubit_op_isa = qubit_op.apply_layout(circuit_isa.layout)

cx_count = circuit_isa.count_ops().get('cx', 0) + circuit_isa.count_ops().get('ecr', 0)
print(f'Transpiled circuit depth: {circuit_isa.depth()}')
print(f'2Q gate count:            {cx_count}')
print(f'Free parameters:          {circuit_isa.num_parameters} (must be 0)')
assert circuit_isa.num_parameters == 0, 'Unbound parameters remain -- check assign_parameters'
print('ASSERTION PASSED: All parameters bound. Circuit ready for hardware.')


## Step 5 — Single-Shot Hardware Estimator Run

One Estimator PUB. One job. One result.
**Save the Job ID immediately** -- it is your IBM-authenticated hardware provenance record.


In [ ]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, Session

print('Opening IBM Quantum session...')
t5 = time.time()

with Session(backend=backend) as session:
    estimator = Estimator(mode=session)
    estimator.options.resilience_level = 1
    estimator.options.default_shots = 8192

    pub = (circuit_isa, qubit_op_isa)
    job = estimator.run([pub])

    job_id = job.job_id()
    print(f'JOB ID: {job_id}')
    print('SAVE THIS JOB ID -- it is your hardware provenance record.')
    print('Waiting for result (typically 30-90 s)...')

    result_hw = job.result()
    e_hw = result_hw[0].data.evs

elapsed = time.time() - t5
err_hw  = abs(e_hw - e_gs) * 1000
err_vs_classical = abs(e_hw - best_energy) * 1000

print('=' * 60)
print(f'HARDWARE RESULT ({backend.name})')
print('=' * 60)
print(f'E (hardware, ZNE):       {e_hw:.6f} Ha')
print(f'E (CASCI reference):     {e_gs:.8f} Ha')
print(f'E (classical VQE best):  {best_energy:.8f} Ha')
print(f'Hardware vs CASCI:       {err_hw:.4f} mHa')
print(f'Hardware vs classical:   {err_vs_classical:.4f} mHa  <- noise overhead')
print(f'Job ID:                  {job_id}')
print(f'Wall time:               {elapsed:.1f}s')
print('=' * 60)


## Step 6 — Retrieve a Past Job by ID

If you need to re-fetch the result after the session closes (e.g., the next day),
use this cell. Results are stored on IBM servers for 90 days.


In [ ]:
# Replace with your actual job ID from Step 5
SAVED_JOB_ID = 'PASTE_JOB_ID_HERE'

past_job    = service.job(SAVED_JOB_ID)
past_result = past_job.result()
e_retrieved = past_result[0].data.evs

print(f'Retrieved job: {SAVED_JOB_ID}')
print(f'Backend:       {past_job.backend().name}')
print(f'Status:        {past_job.status()}')
print(f'Creation time: {past_job.creation_date}')
print(f'Energy:        {e_retrieved:.6f} Ha')
print(f'vs CASCI ref:  {abs(e_retrieved - e_gs)*1000:.4f} mHa')


## Step 7 — Full Result Chain Summary

Print the complete result chain for your paper supplementary material.


In [ ]:
print('NOTEBOOK 10 -- FULL RESULT CHAIN')
print('=' * 62)
print(f'[C1] CASCI(6,6) reference:         {e_casci:.8f} Ha')
print(f'[C2] H_mat ground state:           {e_gs:.8f} Ha  ({abs(e_gs-e_casci)*1000:.6f} mHa)')
print(f'[C3] JW corrected ground state:    {e_jw_fixed:.8f} Ha  ({abs(e_jw_fixed-e_gs)*1000:.6f} mHa)')
print(f'[C4] Classical VQE (best of 5):    {best_energy:.8f} Ha  ({best_err:.4f} mHa)')
print(f'[C5] Hardware ({backend.name}):  {e_hw:.6f} Ha  ({err_hw:.4f} mHa)')
print(f'     Job ID: {job_id}')
print('=' * 62)
print()
print('Frozen-core bug demonstration:')
print(f'  ecore (PySCF naive):  {ecore_pyscf:.8f} Ha')
print(f'  ecore (corrected):    {ecore_needed:.8f} Ha')
print(f'  Discrepancy:          {ecore_needed - ecore_pyscf:.4f} Ha (silent error if uncorrected)')
print()
errs = [str(round(r[2], 4)) for r in all_results]
print(f'VQE seed errors (mHa): {errs}')
print(f'Mean +/- std: {mean_err:.4f} +/- {std_err:.4f} mHa')
